In [ ]:
import spateo as st
import numpy as np
import pandas as pd
import seaborn as sns
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
import matplotlib.colors as mcolors
import dynamo as dyn
import plotly.express as px

In [ ]:
## 聚类结果domain

In [ ]:
adata_bin100 = st.read('data/raw/spatial_mask_file.h5ad')
adata_bin100

In [ ]:
adata_cellbin = st.read('data/processed/spatial_cellbin_file')
adata_cellbin

In [ ]:
adata_bin100.uns['__type'] = 'UMI'
adata_cellbin.uns['__type'] = 'UMI'

In [ ]:
# Layer annotation
st.pl.space(
    adata_bin100,
    color=['region'],
    show_legend="upper left",
    figsize=(4, 3),
    color_key_cmap="tab20"
)

In [ ]:
# 清理坐标数据空值
spatial_coords = adata_cellbin.obsm["spatial"].astype(np.float32)
adata_cellbin.obsm["spatial"] = np.nan_to_num(spatial_coords, nan=0.0)

# 调用函数（确保使用合并后标签）
st.dd.set_domains(
    adata_high_res=adata_cellbin,
    adata_low_res=adata_bin100,
    bin_size_high=1,
    bin_size_low=100,
    cluster_key="region",  # 确保≤20类
    k_size=1.8,
    min_area=16
)

In [ ]:
# View transfered spatial domain
st.pl.space(
    adata=adata_cellbin,
    color=['domain_region'],  # 着色字段
    pointsize=0.03,                     # 调整点大小的正确参数（默认值通常为0.5-1.0）
    show_legend="upper left",
    figsize=(4, 3),
    color_key_cmap="tab20",
#    alpha=0.7                          # 可选：添加透明度避免重叠
)

In [ ]:
adata_cellbin.write("results/spateo/spateo_output_file", compression="gzip")
adata_bin100.write("results/spateo/spateo_output_file", compression="gzip")

In [ ]:
## 手动框定边界

In [ ]:
adata_bin100 = st.read('results/spateo/spateo_output_file')
adata_bin100

In [ ]:
# Extract an area of interest
adata_bin100.obsm['spatial_bin100'] = adata_bin100.obsm['spatial']//100
cluster_label_image_lowres = st.dd.gen_cluster_image(adata_bin100, bin_size=1, spatial_key="spatial_bin100", cluster_key='region', show=True)
cluster_label_list = np.unique(adata_bin100[adata_bin100.obs['region'].isin(['L2','L3','L5','L6']), :].obs["cluster_img_label"])

In [ ]:
contours, cluster_image_close, cluster_image_contour = st.dd.extract_cluster_contours(cluster_label_image_lowres, cluster_label_list, bin_size=1, k_size=6, show=False)
px.imshow(cluster_image_contour, width=500, height=500)

In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(cluster_image_contour, cmap='viridis')

# 标记四个点
pnt_xy =(206,24)  #左上
pnt_xY = (205, 14) #左下
pnt_XY = (71, 15) #右上
pnt_Xy = (94, 15) #右下
points = {
    "pnt_xy (up left)": pnt_xy, 
    "pnt_xY (down left)": pnt_xY,
    "pnt_XY (up right)": pnt_XY,
    "pnt_Xy (down right)": pnt_Xy
}

for label, (x, y) in points.items():
    plt.scatter(x, y, s=100, label=label)
    plt.text(x+2, y+2, f"{label}\n({x},{y})", color='white', 
             bbox=dict(facecolor='red', alpha=0.7))

plt.legend()
plt.title("check")
plt.show()

In [ ]:
import numpy as np
from scipy.spatial import KDTree
import cv2

# 计算轮廓面积并获取最大轮廓索引
contour_areas = [cv2.contourArea(cnt) for cnt in contours]
max_contour_idx = np.argmax(contour_areas)  # 关键：使用此索引

# 将轮廓点转为整数坐标（OpenCV特性）
contour_points = np.vstack(contours[max_contour_idx]).astype(int)
contour_points_set = set(tuple(p) for p in contour_points)

# 容差匹配函数（解决1-2像素偏移）
def find_nearest_point(target_point, point_set, tolerance=2):
    for point in point_set:
        if abs(point[0] - target_point[0]) <= tolerance and abs(point[1] - target_point[1]) <= tolerance:
            return point
    return None

# 修正四个点
pnt_xy_adj = find_nearest_point((206,24), contour_points_set)
pnt_xY_adj = find_nearest_point((207, 14), contour_points_set)
pnt_XY_adj = find_nearest_point((71, 15), contour_points_set)
pnt_Xy_adj = find_nearest_point((94, 15), contour_points_set)

In [ ]:
max_contour_idx

In [ ]:
st.dd.digitize(
    adata=adata_bin100,
    ctrs=contours,
    ctr_idx=0,  # 使用最大轮廓索引
    pnt_xy=pnt_xy_adj,        # 左上
    pnt_xY=pnt_xY_adj,        # 左下
    pnt_Xy=pnt_Xy_adj,        # 右下
    pnt_XY=pnt_XY_adj,        # 右上
    spatial_key="spatial_bin100"
)

In [ ]:
adata_bin100.write("results/spateo/spateo_output_file", compression="gzip")
adata_bin100

In [ ]:
## 可视化热图

In [ ]:
# 设置颜色映射
cmap = plt.cm.viridis  # 你可以选择其他颜色映射

# 绘制热图
fig, ax = plt.subplots(figsize=(5, 5))  # 设置图像大小
sc.pl.spatial(
    adata_bin100,
    img_key=None,  # 如果你有空间图像，可以指定 img_key
    color='digital_layer',
    size=12,  # 点的大小
    spot_size=12,  # 点的大小
    cmap=cmap,
    title='',  # 去除标题
    show=False,  # 不直接显示图像
    ax=ax  # 指定绘图的轴
)

# 去除坐标轴文字
ax.set_xticks([])  # 去除 x 轴刻度
ax.set_yticks([])  # 去除 y 轴刻度
ax.set_xlabel('')  # 去除 x 轴标签
ax.set_ylabel('')  # 去除 y 轴标签

# 保存图像
plt.savefig('results/spateo/spateo_output_file', bbox_inches='tight', dpi=300)
plt.savefig('results/spateo/spateo_output_file', bbox_inches='tight')
plt.show()

In [ ]:
## 绘制有分层标记的概率密度曲线

In [ ]:
# 读数据
#adata_bin100 = sc.read_h5ad('results/spateo/spateo_output_file')
#adata = sc.read_h5ad('results/tacco/tacco_output_file')
adata_bin100 = sc.read_h5ad('results/spateo/spateo_output_file')
adata = sc.read_h5ad('results/tacco/tacco_output_file')

# 把 digital_layer 映射到 adata
common_cells = list(set(adata_bin100.obs_names) & set(adata.obs_names))
digital_map = adata_bin100.obs.loc[common_cells, 'digital_layer'].to_dict()
adata.obs['digital_layer'] = adata.obs_names.map(digital_map).astype(float)

# 准备数据
interested_cluster = 'EX_NTNG1'
bin_width = 10
df = pd.concat([
    adata.obs['digital_layer'].reset_index(drop=True),
    adata.obsm['pred_celltype'][interested_cluster].reset_index(drop=True)
], axis=1, ignore_index=True)
df.columns = ['digital_layer', 'probability']
df = df[df['probability'] > 0]
df['bin'] = (df['digital_layer'] // bin_width) * bin_width

# 分组统计
grouped = df.groupby('bin')['probability'].agg(['mean', 'std'])
min_bin, max_bin = grouped.index.min(), grouped.index.max()
grouped.index = (grouped.index - min_bin) / (max_bin - min_bin) * 100  # 0-100

In [ ]:
# 计算四层中位数（同样归一化）
palette = {'L2': '#ff7f0e', 'L3': '#2ca02c',
           'L5': '#d62728', 'L6': '#9467bd'}
layer_median = (adata_bin100.obs
                .groupby('region')['digital_layer']
                .median()                 # ← 这里换成 median
                .reindex(['L6', 'L5', 'L3', 'L2'])
                .dropna())
norm_median = (layer_median - min_bin) / (max_bin - min_bin) * 100

In [ ]:
# 绘图
fig, ax = plt.subplots(figsize=(10, 6))

# 曲线 + 误差带
ax.plot(grouped.index, grouped['mean'], color='b', lw=2)
ax.fill_between(grouped.index,
                grouped['mean'] - grouped['std'],
                grouped['mean'] + grouped['std'],
                color='b', alpha=0.2)

# 四层中位数竖线
for region in ['L6', 'L5', 'L3', 'L2']:
    ax.axvline(norm_median[region], color=palette[region], ls='--', lw=2.5)

# ---- 样式调整 ----
ax.grid(False)                      # 去掉网格
ax.set_xlabel('')                   # 去掉横轴文字
ax.set_ylabel('')                   # 去掉纵轴文字
ax.tick_params(labelsize=12)        # 保留刻度数字大小

# 仅保留黑色边框
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(1)

# 7. 保存
out_dir = '/data/work/Spateo_digital/'
fig.savefig(out_dir + 'EX_NTNG1_Y00782J8_with_layerLines_median_clean.png', dpi=300)
fig.savefig(out_dir + 'EX_NTNG1_Y00782J8_with_layerLines_median_clean.pdf')
plt.show()